## 简介

Middleware(中间件)，简单说就是Agent 执行过程中的钩子函数，是 LangChain 1.x 的“王牌”工程化能力。

钩子是框架或系统在某些关键执行点暴露的扩展接口。开发者可以“挂上”自己的逻辑，在那些点上插入、修改或替换行为，而无需改变主流程代码。就像在流水线上某个环节设置了一个“检查点”或“插入器”。

借助中间件，开发者可以高度定制和控制Agent运行的每一个环节，是处理 **Agent 生命周期**的标准方式。

![中间件示意图](../Pics/1.png)


## 作用

为什么需要中间件

如果没有中间件，Agent 的执行流程通常比较直接：
> 1 用户输入 → 拼接提示词/消息 → 调用模型 → 如有需要调用工具 → 返回结果

这种方式对于简单场景已经足够，但一旦进入真实项目，往往会遇到很多额外需求，例如：
- 想根据问题复杂度动态**切换模型**；
- 想**限制**某些用户只能调用部分工具；
- 想在工具报错时**自动重试**或返回兜底结果；
- 想在模型调用前**插入额外的系统提示**；
- 想记录每一步的**执行日志**，方便排查问题；
- 想在敏感信息出现时**阻断执行**；
- 想在正式执行工具前增加**人工审批**。

---

这些需求有一个共同特点：它们不是 Agent 的核心业务逻辑，但又会影响 Agent 的执行过程。如果把这些逻辑全部直接写进主流程，会带来几个问题：
### 1) 主流程会迅速变乱
Agent 本身只需要关心“理解用户需求、决定是否调用工具、生成结果”，
但一旦把日志、鉴权、重试、风控、审计都塞进去，主逻辑就会变得臃肿。

### 2) 很多逻辑是横切需求，难以复用
例如日志、重试、风控、权限控制，通常不是某一个 Agent 独有的，而是多个 Agent 都需要。如果直接写死在每个 Agent 里，会产生大量重复代码。

### 3) 流程控制粒度不够细
有些逻辑必须发生在“模型调用前”，有些要发生在“工具调用后”，如果没有统一的执行拦截点，开发者只能手动改主流程，既麻烦又容易出错。

### 4) 后期维护成本高
当你需要增加一个新规则，例如“所有外部工具调用前都先做审计”，如果系统没有中间件机制，往往需要修改很多处代码。

## 总结：
中间件的价值就在于把这些与业务无关、但与执行过程强相关的横切逻辑，从 Agent 主流程中分离出来。让Agent主体代码**聚焦业务**，而借助中间件，实现**拦截流程、修改流程、增强流程**。

简言之，LangChain 1.x 的中间件能实现如下功能：
- ✅ 日志与分析 - 追踪行为、调试、性能监控
- ✅ 转换 - 修改提示词、工具选择、输出格式
- ✅ 容错 - 重试、降级、早期终止
- ✅ 安全 - 限流、守护规则、PII检测

## 分类

根据LangChain是否已经定义了来分类：
- **自定义中间件**：允许开发者自定义，从而实现更加灵活的Agent行为管理
- **内置中间件**：LangChain实现并提供的
  - 模型供应商定制的中间件：依赖于特定模型服务的实现（不是本课的重点）
  - 和模型供应商无关的中间件。LangChain提供的与供应商无关的中间件如下：

链接：https://docs.langchain.com/oss/python/langchain/middleware/overview

Provider-agnostic middleware

以下中间件可与任何LLM提供商配合使用：

| 中间件 | 描述 |
| ---- | ---- |
| Summarization | 当接近token限制时自动总结对话历史。 |
| Human-in-the-loop | 暂停执行以等待人工审批工具调用。 |
| Model call limit | 限制模型调用次数以防止过度消耗成本。 |
| Tool call limit | 通过限制调用次数来控制工具执行。 |
| Model fallback | 当主模型失败时自动回退到备用模型。 |
| PII detection | 检测和处理个人身份信息（PII）。 |
| To-do list | 为Agent配备任务规划和跟踪能力。 |
| LLM tool selector | 在调用主模型之前使用LLM选择相关工具。 |
| Tool retry | 使用指数退避自动重试失败的工具调用。 |
| Model retry | 使用指数退避自动重试失败的模型调用。 |
| LLM tool emulator | 使用LLM模拟工具执行以用于测试目的。 |
| Context editing | 通过修剪或清除工具使用来管理对话上下文。 |
| Shell tool | 向Agent暴露持久化的shell会话以执行命令。 |
| File search | 提供针对文件系统文件的Glob和Grep搜索工具。 |
| Filesystem | 为Agent提供用于存储上下文和长期记忆的文件系统。 |
| Subagent | 添加生成子Agent的能力。 |


LangChain提供的和模型供应商无关的内置中间件分为六个类别

### 类型1：成本与资源控制类
**核心目标**：控成本、控配额、避免无限调用
这类中间件主要解决“Agent太贵、太能跑、停不下来”的问题。
包含：
- Model call limit：限制模型调用次数，防止一次任务反复请求LLM，导致费用失控
- Tool call limit：限制工具调用次数，避免Agent无限试错、死循环调工具
- Summarization：在上下文快满时自动总结历史，减少token消耗
- Context editing：裁剪上下文、清理工具调用痕迹，本质上也是为了节省上下文成本

**业务场景理解**：
适合生产环境的成本治理、配额治理、长会话优化、SaaS产品控费。

### 类型2：稳定性与容错保障类
**核心目标**：保证服务不中断、失败后尽量自动恢复
这类中间件主要解决“调用失败怎么办、模型挂了怎么办、工具超时怎么办”。
包含：
- Model fallback：主模型失败时切换备用模型
- Model retry：模型调用失败后自动重试
- Tool retry：工具调用失败后自动重试

**业务场景理解**：
适合线上生产系统，尤其是多模型、多工具依赖的Agent。
本质上是在做高可用、容灾、鲁棒性建设。

### 类型3：安全与合规风控类
**核心目标**：让Agent可控、可审、合规
这类中间件主要解决“Agent乱执行、泄露敏感信息、做危险操作”的问题。
包含：
- Human-in-the-loop：在关键工具调用前暂停，等人人工审批
- PII detection：检测和处理个人敏感信息
- Model call limit / Tool call limit：某种意义上也可归到风控，因为它能防止异常滥用

**业务场景理解**：
适合企业内部系统、客服系统、审批流、数据查询类Agent。
尤其是涉及：发邮件、调数据库、调财务/人事系统、导出敏感信息、执行外部动作等

### 类型4：决策增强与智能编排类
**核心目标**：提升Agent的决策质量和任务拆解能力
这类中间件主要解决“Agent不够聪明、不会规划、不会先筛工具”的问题。
包含：
- To-do list：给Agent增加任务规划、分步骤执行和状态跟踪能力
- LLM tool selector：当工具太多时，用子模型筛选最相关的几个工具交给主模型
- Subagent：允许生成子Agent，把复杂任务拆给不同角色处理

**业务场景理解**：
适合复杂任务流，比如：研究型Agent、多步骤分析、报告生成、多角色协作、长链路任务编排等。
这类本质上是在增强Agent的“脑子”与“组织能力”。

### 类型5：执行能力扩展类
**核心目标**：给Agent更多“手脚”
这类中间件主要解决“Agent只能聊天，不能真正操作环境”的问题。
包含：
- Shell tool：给Agent持久shell，会执行命令
- File search：给Agent文件搜索能力，能做Glob/Grep
- Filesystem：给Agent文件系统读写与长期存储能力

**业务场景理解**：
适合工程Agent、代码Agent、本地自动化Agent、运维Agent。
本质上是把Agent从“纯推理”扩展成“能操作环境的执行体”。

### 类型6：开发调试与测试辅助类
**核心目标**：方便开发、测试、验证Agent行为
这类中间件主要不是直接服务业务，而是服务于研发和调试阶段。
包含：
- LLM tool emulator：用LLM模拟工具执行，便于测试（最典型）
- Summarization：有时也可辅助调试长会话表现
- Context editing：可用于测试上下文裁剪效果
- Human-in-the-loop：也常用于调试高风险步骤

**业务场景理解**：
适合开发阶段快速验证流程、做mock、减少真实工具依赖。
